## installation command (if packages not already downloaded)



In [6]:
!pip install matplotlib
!pip install pandas
!pip install statsmodels
!pip install sklearn
!pip install pmdarima -q

  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


# Libraries import

for this  sort of task i decided to move with time series modelingm method Sarima

In [7]:
import numpy as np
import pandas as pd
import matplotlib

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
import itertools
matplotlib.use('Agg')
warnings.filterwarnings('ignore')
import pmdarima as pm
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.metrics import mean_squared_error

# Data import and splitting before training

Data is split in a reasonable fashion, we have 18 years of data, we use 16 for training and 2 for validation. Since the observations are daily it is big enough sample size

In [8]:
df = pd.read_csv('temperature.csv')
df['date'] = pd.to_datetime(df['date'], dayfirst=True)
df = df.sort_values('date').reset_index(drop=True)
df = df.set_index('date')
series = df['T2M'].astype(float)


#Train validation split
train = series[series.index.year <= 2016]
val   = series[series.index.year.isin([2017, 2018])]

# Fourier transformation of calendar dates

Fourier transformation utilises combination of sine and cosine to create a circle out of the date of the year. This is used to model the inherent seasonality of the dataset. We create multiple pair for sines and cosines to even get som possible seasonalities smaller than 365 days.

In [16]:
# Fourier terms capture the 365-day cycle without a massive state vector
# S=365 in SARIMAX is infeasible (OOM) — this is the standard workaround
K = 5  # number of sin/cos pairs

def make_fourier(n, period=365, K=5):
    t = np.arange(1, n + 1)
    return np.column_stack([
        fn(2 * np.pi * k * t / period)
        for k in range(1, K + 1)
        for fn in (np.sin, np.cos)
    ])

fourier_train = make_fourier(len(train), period=365, K=K)

# Arima parameters choosing

To address non-stationarity we difference the series once (d=1). Rather than using seasonal ARIMA with a 365-day period — which is computationally infeasible due to the massive state vector the Kalman filter must maintain — we capture the yearly cycle through Fourier terms (K=5 sin/cos pairs at harmonics of 365 days) passed as exogenous regressors. This is the standard workaround for long seasonal periods and is equally justified by the expectation that weather patterns repeat on a yearly time frame.

For model selection we use auto_arima with stepwise search over the non-seasonal ARIMA orders, as the limited computational capacity available on Colab rules out both exhaustive grid search and direct seasonal fitting with S=365.

In [17]:
auto_model = pm.auto_arima(
    train,
    exogenous=fourier_train,
    d=1, D=0,
    seasonal=False,
    start_p=0, max_p=3,
    start_q=0, max_q=3,
    stepwise=True,
    suppress_warnings=True,
    error_action="ignore",
    trace=True,
    maxiter=200,
    enforce_stationarity=False,
    enforce_invertibility=False,
)

# preserve downstream variable names
best_order          = auto_model.order
best_seasonal_order = (0, 0, 0, 0)
best_aic            = auto_model.aic()

print(f" Best: ARIMA{best_order} + Fourier(K={K})  AIC={best_aic:.2f}")


Performing stepwise search to minimize aic
 ARIMA(0,1,0)(0,0,0)[0] intercept   : AIC=29386.271, Time=0.04 sec
 ARIMA(1,1,0)(0,0,0)[0] intercept   : AIC=29386.649, Time=0.11 sec
 ARIMA(0,1,1)(0,0,0)[0] intercept   : AIC=29384.539, Time=0.14 sec
 ARIMA(0,1,0)(0,0,0)[0]             : AIC=29384.271, Time=0.04 sec
 ARIMA(1,1,1)(0,0,0)[0] intercept   : AIC=28809.457, Time=0.41 sec
 ARIMA(2,1,1)(0,0,0)[0] intercept   : AIC=28520.521, Time=0.44 sec
 ARIMA(2,1,0)(0,0,0)[0] intercept   : AIC=28886.255, Time=0.20 sec
 ARIMA(3,1,1)(0,0,0)[0] intercept   : AIC=28504.255, Time=0.73 sec
 ARIMA(3,1,0)(0,0,0)[0] intercept   : AIC=28791.469, Time=0.28 sec
 ARIMA(3,1,2)(0,0,0)[0] intercept   : AIC=28499.193, Time=0.89 sec
 ARIMA(2,1,2)(0,0,0)[0] intercept   : AIC=28497.852, Time=0.62 sec
 ARIMA(1,1,2)(0,0,0)[0] intercept   : AIC=28497.695, Time=0.40 sec
 ARIMA(0,1,2)(0,0,0)[0] intercept   : AIC=28626.123, Time=0.21 sec
 ARIMA(1,1,3)(0,0,0)[0] intercept   : AIC=28498.160, Time=0.71 sec
 ARIMA(0,1,3)(0,0,0

# Chosen model creation

Here we simply use the best established model and predict for years 2019 and 2020

In [18]:
model_train = SARIMAX(
    train,
    exog=fourier_train,
    order=best_order,
    seasonal_order=best_seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False
)
fit_train = model_train.fit(disp=False, maxiter=500)

print(fit_train.summary())

# Forecast validation period
fourier_val = make_fourier(len(val), period=365, K=K)
# shift phase to continue from end of training
t_offset = len(train)
t_val = np.arange(t_offset + 1, t_offset + len(val) + 1)
fourier_val = np.column_stack([
    fn(2 * np.pi * k * t_val / 365)
    for k in range(1, K + 1)
    for fn in (np.sin, np.cos)
])

fc_val    = fit_train.forecast(steps=len(val), exog=fourier_val)
val_mse   = mean_squared_error(val.values, fc_val.values)
val_rmse  = np.sqrt(val_mse)
print(f" MSE  = {val_mse:.4f}")
print(f" RMSE = {val_rmse:.4f}")


                               SARIMAX Results                                
Dep. Variable:                    T2M   No. Observations:                 6210
Model:               SARIMAX(1, 1, 2)   Log Likelihood              -14022.398
Date:                Fri, 27 Mar 2026   AIC                          28072.796
Time:                        19:30:00   BIC                          28167.062
Sample:                    01-01-2000   HQIC                         28105.476
                         - 12-31-2016                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
x1            -6.1442      0.187    -32.913      0.000      -6.510      -5.778
x2           -10.6920      0.181    -59.021      0.000     -11.047     -10.337
x3             0.1437      0.145      0.990      0.3

In [11]:
#fit model on the whole dataset and forecast 2019 and 2020
fourier_full = make_fourier(len(series), period=365, K=K)

model_full = SARIMAX(
    series,
    exog=fourier_full,
    order=best_order,
    seasonal_order=best_seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False
)
fit_full = model_full.fit(disp=False, maxiter=500)

n_forecast   = 731   # 2019 (365) + 2020 (366 leap)
future_dates = pd.date_range('2019-01-01', periods=n_forecast, freq='D')

# Fourier terms for the forecast horizon
t_offset = len(series)
t_future = np.arange(t_offset + 1, t_offset + n_forecast + 1)
fourier_future = np.column_stack([
    fn(2 * np.pi * k * t_future / 365)
    for k in range(1, K + 1)
    for fn in (np.sin, np.cos)
])

fc_result    = fit_full.get_forecast(steps=n_forecast, exog=fourier_future)
fc_mean      = fc_result.predicted_mean
#confidence intervals useful in visualization
fc_ci        = fc_result.conf_int(alpha=0.05)
fc_mean.index = future_dates
fc_ci.index   = future_dates

df_forecast = pd.DataFrame({
    'date':          future_dates,
    'T2M_forecast':  np.round(fc_mean.values, 4),
    'lower_95':      np.round(fc_ci.iloc[:, 0].values, 4),
    'upper_95':      np.round(fc_ci.iloc[:, 1].values, 4),
})

#saving forecasted values
df_forecast.to_csv('sarima_forecast_2019_2020.csv', index=False)


# Model evaluation(ljubgbox, etc)

In this part we make sure to test for autocorrelation in residuals to check if model is correctly specified or if other measures are needed.

In [12]:
#testing residuals for futher autocorrelation
resid      = fit_full.resid
lb_result  = acorr_ljungbox(resid, lags=[20], return_df=True)
lb_stat    = lb_result['lb_stat'].values[0]
lb_pval    = lb_result['lb_pvalue'].values[0]
print(f"Ljung-Box Q(20) = {lb_stat:.2f}  p = {lb_pval:.4f}")
print(f"({'white noise' if lb_pval > 0.05 else 'autocorrelation'})")

# acf of residuals (manual, for plotting)
def acf_manual(x, max_lag=50):
    n = len(x); mean = x.mean(); var = np.var(x)
    return [np.mean((x[:n-k] - mean) * (x[k:] - mean)) / var
            for k in range(1, max_lag+1)]

resid_acf = acf_manual(resid.values, max_lag=50)
conf_band = 1.96 / np.sqrt(len(resid))

Ljung-Box Q(20) = 41.59  p = 0.0031
(autocorrelation)


# Visualizations
Visualization provides more intuitive way of observing the results of previous efforts.

Following visualization created with assistance of artificial intelligence(claude was instructed to provide visualization, so only code provided)

In [15]:
DARK_BG  = '#0f1117'
PANEL_BG = '#1a1d27'
BLUE     = '#4f8ef7'
ORANGE   = '#ff6b35'
GREEN    = '#00c896'
CYAN     = '#00d4ff'
PURPLE   = '#b07fff'
TEXT     = '#e0e0e0'
GRID     = '#2a2d3a'

def style_ax(ax, title=''):
    ax.set_facecolor(PANEL_BG)
    ax.tick_params(colors=TEXT, labelsize=9)
    for spine in ax.spines.values(): spine.set_color(GRID)
    ax.grid(axis='y', color=GRID, linewidth=0.5, linestyle='--')
    ax.grid(axis='x', color=GRID, linewidth=0.3, linestyle=':')
    if title:
        ax.set_title(title, color=TEXT, fontsize=11, fontweight='bold', pad=10)



p_ord, d_ord, q_ord = best_order
# Fourier approach — no seasonal ARIMA order

fig = plt.figure(figsize=(20, 26), facecolor=DARK_BG)
gs  = gridspec.GridSpec(4, 2, figure=fig, hspace=0.46, wspace=0.28)
fig.suptitle(
    f'ARIMA({p_ord},{d_ord},{q_ord}) + Fourier(K={K}) — FIPS 36061',
    color='white', fontsize=15, fontweight='bold', y=0.99
)

#Panel 1: Full history + forecast with CI
ax1 = fig.add_subplot(gs[0, :])
style_ax(ax1, 'Historical (2000–2018) + ARIMA+Fourier Forecast (2019–2020) with 95% CI')

hist_mo = series.resample('ME').mean()
fc_mo   = fc_mean.resample('ME').mean()
ci_lo   = fc_ci.iloc[:, 0].resample('ME').mean()
ci_hi   = fc_ci.iloc[:, 1].resample('ME').mean()

ax1.plot(series.index, series.values, color=BLUE,   alpha=0.15, linewidth=0.4)
ax1.plot(hist_mo.index, hist_mo.values, color=BLUE,  linewidth=2.0, label='Historical (monthly avg)')
ax1.plot(future_dates, fc_mean.values,  color=ORANGE, alpha=0.20, linewidth=0.4)
ax1.plot(fc_mo.index,  fc_mo.values,    color=ORANGE, linewidth=2.4, label='Forecast (monthly avg)')
ax1.fill_between(fc_mo.index, ci_lo.values, ci_hi.values, color=ORANGE, alpha=0.15, label='95% CI')
ax1.axvline(pd.Timestamp('2019-01-01'), color='white', linestyle='--', linewidth=1.0, alpha=0.6)
ax1.set_ylabel('Temperature (°C)', color=TEXT)
ax1.legend(facecolor=PANEL_BG, labelcolor=TEXT, fontsize=10)

#Panel 2: Validation actual vs predicted
ax2 = fig.add_subplot(gs[1, 0])
style_ax(ax2, 'Validation: Actual vs Predicted (2017–2018)')

act_mo  = val.resample('ME').mean()
pred_mo = pd.Series(fc_val.values, index=val.index).resample('ME').mean()

ax2.plot(act_mo.index,  act_mo.values,  color=BLUE,  linewidth=2.2, label='Actual')
ax2.plot(pred_mo.index, pred_mo.values, color=GREEN, linewidth=2.2,
         linestyle='--', label='Predicted')
ax2.fill_between(act_mo.index, act_mo.values, pred_mo.values, alpha=0.15, color=GREEN)
ax2.set_ylabel('Temperature (°C)', color=TEXT)
ax2.legend(facecolor=PANEL_BG, labelcolor=TEXT, fontsize=9)
ax2.text(0.04, 0.90,
         f'MSE  = {val_mse:.3f}\nRMSE = {val_rmse:.3f}',
         transform=ax2.transAxes, color=CYAN, fontsize=10,
         bbox=dict(facecolor=DARK_BG, edgecolor=GRID, boxstyle='round,pad=0.4'))

#Panel 3: Residuals
ax3 = fig.add_subplot(gs[1, 1])
style_ax(ax3, 'Model Residuals')

r = resid.values
ax3.plot(r, color=CYAN, alpha=0.55, linewidth=0.5)
ax3.axhline(0,          color='white', linewidth=0.8, linestyle='--', alpha=0.6)
ax3.axhline( 2*r.std(), color=ORANGE, linewidth=0.8, linestyle=':', alpha=0.7, label='±2σ')
ax3.axhline(-2*r.std(), color=ORANGE, linewidth=0.8, linestyle=':', alpha=0.7)
ax3.set_xlabel('Time index', color=TEXT)
ax3.set_ylabel('Residual', color=TEXT)
ax3.legend(facecolor=PANEL_BG, labelcolor=TEXT, fontsize=9)

#Panel 4: Residual ACF
ax4 = fig.add_subplot(gs[2, 0])
style_ax(ax4, f'Residual ACF  (Ljung-Box Q(20) = {lb_stat:.1f}, p = {lb_pval:.3f})')

lags = np.arange(1, len(resid_acf) + 1)
bar_colors = [GREEN if abs(v) < conf_band else ORANGE for v in resid_acf]
ax4.bar(lags, resid_acf, color=bar_colors, alpha=0.75, width=0.7)
ax4.axhline( conf_band, color='white', linewidth=1.0, linestyle='--', alpha=0.7, label='95% CI')
ax4.axhline(-conf_band, color='white', linewidth=1.0, linestyle='--', alpha=0.7)
ax4.axhline(0,          color='white', linewidth=0.5, alpha=0.3)
ax4.set_xlabel('Lag', color=TEXT)
ax4.set_ylabel('ACF', color=TEXT)
ax4.legend(facecolor=PANEL_BG, labelcolor=TEXT, fontsize=9)

#Panel 5: Residual distribution
ax5 = fig.add_subplot(gs[2, 1])
style_ax(ax5, 'Residual Distribution')

counts, bins = np.histogram(r, bins=50)
ax5.bar(bins[:-1], counts, width=np.diff(bins), color=PURPLE, alpha=0.75, edgecolor=DARK_BG)
mu, sigma = r.mean(), r.std()
x_norm    = np.linspace(bins[0], bins[-1], 200)
pdf       = (counts.sum() * np.diff(bins).mean()) * \
            np.exp(-0.5 * ((x_norm - mu) / sigma)**2) / (sigma * np.sqrt(2 * np.pi))
ax5.plot(x_norm, pdf, color=ORANGE, linewidth=2.0, label=f'N({mu:.2f}, {sigma:.2f}²)')
ax5.set_xlabel('Residual value', color=TEXT)
ax5.set_ylabel('Count', color=TEXT)
ax5.legend(facecolor=PANEL_BG, labelcolor=TEXT, fontsize=9)

#Panel 6: Monthly bars — historical vs 2019 vs 2020
ax6 = fig.add_subplot(gs[3, :])
style_ax(ax6, 'Monthly Forecast Averages vs Historical Climatology')

fc_2019   = df_forecast[df_forecast['date'].dt.year == 2019]
fc_2020   = df_forecast[df_forecast['date'].dt.year == 2020]
months    = range(1, 13)
mlabels   = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

hist_clim = [series[series.index.month == m].mean() for m in months]
avg_2019  = [fc_2019[fc_2019['date'].dt.month == m]['T2M_forecast'].mean() for m in months]
avg_2020  = [fc_2020[fc_2020['date'].dt.month == m]['T2M_forecast'].mean() for m in months]

x     = np.arange(12)
w_bar = 0.26
ax6.bar(x - w_bar, hist_clim, w_bar, color=BLUE,   label='Historical avg (2000–2018)',
        alpha=0.80, edgecolor=DARK_BG)
ax6.bar(x,         avg_2019,  w_bar, color=GREEN,  label='Forecast 2019',
        alpha=0.85, edgecolor=DARK_BG)
ax6.bar(x + w_bar, avg_2020,  w_bar, color=ORANGE, label='Forecast 2020',
        alpha=0.85, edgecolor=DARK_BG)
ax6.set_xticks(x)
ax6.set_xticklabels(mlabels, color=TEXT, fontsize=9)
ax6.set_ylabel('Avg Temperature (°C)', color=TEXT)
ax6.axhline(0, color='white', linewidth=0.5, linestyle='--', alpha=0.4)
ax6.legend(facecolor=PANEL_BG, labelcolor=TEXT, fontsize=10)

plt.savefig('arima_fourier_forecast.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
print("Plot saved → arima_fourier_forecast.png")

#Summary
print(f"""
╔══════════════════════════════════════════════════════════════╗
║  ARIMA({p_ord},{d_ord},{q_ord}) + Fourier(K={K}) — RESULTS SUMMARY
╠══════════════════════════════════════════════════════════════╣
║  AIC               : {fit_full.aic:.2f}
║  BIC               : {fit_full.bic:.2f}
║  Log-Likelihood    : {fit_full.llf:.2f}
╠══════════════════════════════════════════════════════════════╣
║  Validation MSE    : {val_mse:.4f}
║  Validation RMSE   : {val_rmse:.4f}
║  Ljung-Box Q(20)   : {lb_stat:.2f}  (p = {lb_pval:.4f})
╠══════════════════════════════════════════════════════════════╣""")
for yr in [2019, 2020]:
    ydf = df_forecast[df_forecast['date'].dt.year == yr]
    print(f"║  {yr}: mean={ydf['T2M_forecast'].mean():6.2f}°C | "
          f"min={ydf['T2M_forecast'].min():6.2f}°C | "
          f"max={ydf['T2M_forecast'].max():6.2f}°C")
print("╚══════════════════════════════════════════════════════════════╝")

Plot saved → arima_fourier_forecast.png

╔══════════════════════════════════════════════════════════════╗
║  ARIMA(1,1,2) + Fourier(K=5) — RESULTS SUMMARY
╠══════════════════════════════════════════════════════════════╣
║  AIC               : 31476.68
║  BIC               : 31572.50
║  Log-Likelihood    : -15724.34
╠══════════════════════════════════════════════════════════════╣
║  Validation MSE    : 13.1202
║  Validation RMSE   : 3.6222
║  Ljung-Box Q(20)   : 41.59  (p = 0.0031)
╠══════════════════════════════════════════════════════════════╣
║  2019: mean= 11.94°C | min= -0.73°C | max= 24.55°C
║  2020: mean= 11.90°C | min= -0.73°C | max= 24.55°C
╚══════════════════════════════════════════════════════════════╝
